In [1]:
# NORTHSTAR -- Cloud Sync & Twin Features QA for Solace Browser
# DNA: sync(cloud) = crud(schedules) x state(5-vars) x error(graceful) x tunnel(consent) x js(quality) x security(cors+https)
# Probes schedule-cloud.js, schedule-core.js, yinyang-rail.js, tunnel-connect.html, machine-dashboard.html
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "cloud-sync-qa"
NOTEBOOK_PATH = "notebooks/qa/32-cloud-sync-qa.ipynb"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER_ROOT / "web"
JS = WEB / "js"
CSS = WEB / "css"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print("Cloud Sync & Twin Features QA")
assert BROWSER_ROOT.exists()

Cloud Sync & Twin Features QA


In [2]:
# P1: Cloud sync JS files exist and have proper structure
# schedule-cloud.js must have fetch calls with error handling, use HTTPS URLs
print("=" * 60)
print("P1: Cloud Sync JS Structure")
print("=" * 60)

cloud_js = read_file(JS / "schedule-cloud.js")
record("P1.1", "schedule-cloud.js exists and is non-empty",
       len(cloud_js) > 100,
       detail=f"{len(cloud_js)} bytes" if cloud_js else "FILE MISSING")

# Must have fetch calls (cloudFetch wrapper or direct fetch)
has_fetch = "fetch(" in cloud_js or "cloudFetch(" in cloud_js
record("P1.2", "schedule-cloud.js contains fetch calls",
       has_fetch,
       detail="Found fetch/cloudFetch usage" if has_fetch else "No fetch calls found")

# Must have error handling around fetch (catch blocks)
has_catch = ".catch(" in cloud_js or "catch(" in cloud_js
record("P1.3", "schedule-cloud.js has error handling (.catch or catch)",
       has_catch,
       detail="Error handling present" if has_catch else "No .catch() found — network errors unhandled")

# Must use HTTPS URLs (no plain HTTP to external hosts)
http_urls = re.findall(r"http://(?!127\.0\.0\.1|localhost)[^\s'\"]+", cloud_js)
record("P1.4", "No hardcoded plain HTTP URLs to external hosts",
       len(http_urls) == 0,
       detail=f"Found insecure URLs: {http_urls}" if http_urls else "All external URLs use HTTPS or localhost")

# Check that default cloud API URL is HTTPS
https_default = re.search(r"https://.*solaceagi\.com", cloud_js)
record("P1.5", "Default cloud API URL uses HTTPS",
       https_default is not None,
       detail=f"Found: {https_default.group(0)}" if https_default else "No HTTPS solaceagi.com URL found")

# Check schedule-core.js exists
core_js = read_file(JS / "schedule-core.js")
record("P1.6", "schedule-core.js exists and is non-empty",
       len(core_js) > 100,
       detail=f"{len(core_js)} bytes" if core_js else "FILE MISSING")

# Check yinyang-rail.js exists
rail_js = read_file(JS / "yinyang-rail.js")
record("P1.7", "yinyang-rail.js exists and is non-empty",
       len(rail_js) > 100,
       detail=f"{len(rail_js)} bytes" if rail_js else "FILE MISSING")

print(f"\nP1 complete: {sum(1 for r in results if r['status']=='PASS')}/{len(results)} passed")

P1: Cloud Sync JS Structure
PASS: schedule-cloud.js exists and is non-empty -- 10575 bytes
PASS: schedule-cloud.js contains fetch calls -- Found fetch/cloudFetch usage
PASS: schedule-cloud.js has error handling (.catch or catch) -- Error handling present
PASS: No hardcoded plain HTTP URLs to external hosts -- All external URLs use HTTPS or localhost
PASS: Default cloud API URL uses HTTPS -- Found: https://www.solaceagi.com
PASS: schedule-core.js exists and is non-empty -- 24856 bytes
PASS: yinyang-rail.js exists and is non-empty -- 23211 bytes

P1 complete: 7/7 passed


In [3]:
# P2: Schedule core has CRUD operations (create/read/update/delete schedule)
print("=" * 60)
print("P2: Schedule CRUD Operations")
print("=" * 60)

cloud_js = read_file(JS / "schedule-cloud.js")
core_js = read_file(JS / "schedule-core.js")
combined = cloud_js + "\n" + core_js

# Cloud CRUD — schedule-cloud.js should expose create/update/delete
crud_functions = {
    "create": re.search(r"function\s+createCloudSchedule|fn\.createCloudSchedule", cloud_js),
    "load/read": re.search(r"function\s+loadCloudSchedules|fn\.loadCloudSchedules", cloud_js),
    "update": re.search(r"function\s+updateCloudSchedule|fn\.updateCloudSchedule", cloud_js),
    "delete": re.search(r"function\s+deleteCloudSchedule|fn\.deleteCloudSchedule", cloud_js),
}

for op, match in crud_functions.items():
    record(f"P2.{list(crud_functions).index(op)+1}",
           f"Cloud schedule {op} operation exists in schedule-cloud.js",
           match is not None,
           detail=f"Found {match.group(0)}" if match else f"Missing {op} function")

# Verify HTTP methods match CRUD semantics
has_post = "'POST'" in cloud_js or '"POST"' in cloud_js
has_patch = "'PATCH'" in cloud_js or '"PATCH"' in cloud_js
has_delete = "'DELETE'" in cloud_js or '"DELETE"' in cloud_js
record("P2.5", "Cloud CRUD uses correct HTTP methods (POST/PATCH/DELETE)",
       has_post and has_patch and has_delete,
       detail=f"POST={has_post}, PATCH={has_patch}, DELETE={has_delete}")

# Core should have loadActivities (the read path for local data)
has_load_activities = "function loadActivities" in core_js or "fn.loadActivities" in core_js
record("P2.6", "schedule-core.js has loadActivities (local read path)",
       has_load_activities,
       detail="loadActivities found" if has_load_activities else "Missing loadActivities")

# Cloud should merge into local state
has_merge = "mergeCloudData" in cloud_js
record("P2.7", "Cloud data merges into local state (mergeCloudData)",
       has_merge,
       detail="mergeCloudData found" if has_merge else "No merge logic — cloud/local data may not sync")

p2_pass = sum(1 for r in results if r['id'].startswith('P2') and r['status'] == 'PASS')
p2_total = sum(1 for r in results if r['id'].startswith('P2'))
print(f"\nP2 complete: {p2_pass}/{p2_total} passed")

P2: Schedule CRUD Operations
PASS: Cloud schedule create operation exists in schedule-cloud.js -- Found function createCloudSchedule
PASS: Cloud schedule load/read operation exists in schedule-cloud.js -- Found function loadCloudSchedules
PASS: Cloud schedule update operation exists in schedule-cloud.js -- Found function updateCloudSchedule
PASS: Cloud schedule delete operation exists in schedule-cloud.js -- Found function deleteCloudSchedule
PASS: Cloud CRUD uses correct HTTP methods (POST/PATCH/DELETE) -- POST=True, PATCH=True, DELETE=True
PASS: schedule-core.js has loadActivities (local read path) -- loadActivities found
PASS: Cloud data merges into local state (mergeCloudData) -- mergeCloudData found

P2 complete: 7/7 passed


In [4]:
# P3: YinYang rail has state management (IDLE, RUNNING, PREVIEW_READY, BLOCKED, ERROR, DONE)
print("=" * 60)
print("P3: YinYang Rail State Management")
print("=" * 60)

rail_js = read_file(JS / "yinyang-rail.js")

# Check for FSM state handling
expected_states = ["idle", "processing", "error", "blocked", "done"]
# Also check for preview_ready which may appear in message handling
extended_states = expected_states + ["preview_ready"]

found_states = []
missing_states = []
for state in expected_states:
    if state in rail_js.lower():
        found_states.append(state)
    else:
        missing_states.append(state)

record("P3.1", f"YinYang rail handles core FSM states ({len(found_states)}/{len(expected_states)})",
       len(found_states) >= 4,
       detail=f"Found: {found_states}" + (f", Missing: {missing_states}" if missing_states else ""))

# Check for state-to-CSS-class mapping (dot classes)
has_dot_mapping = re.search(r"(idle|processing|error|blocked|done).*--", rail_js)
record("P3.2", "FSM states map to CSS classes (visual indicators)",
       has_dot_mapping is not None,
       detail="State-to-CSS dot mapping found" if has_dot_mapping else "No CSS state mapping")

# Check _applyTopRailState function exists
has_top_rail = "_applyTopRailState" in rail_js or "applyTopRailState" in rail_js
record("P3.3", "Top rail state application function exists",
       has_top_rail,
       detail="_applyTopRailState found" if has_top_rail else "Missing top rail state handler")

# Check for message listener (FSM updates from server/Playwright)
has_message_listener = "addEventListener('message'" in rail_js or 'addEventListener("message"' in rail_js
record("P3.4", "YinYang rail listens for postMessage FSM updates",
       has_message_listener,
       detail="window.message listener found" if has_message_listener else "No postMessage listener")

# Check yy_state message type handling
has_yy_state = "yy_state" in rail_js
record("P3.5", "Handles yy_state message type for FSM transitions",
       has_yy_state,
       detail="yy_state handler found" if has_yy_state else "Missing yy_state message handling")

# Check for auto-expand on approval-required states (preview_ready, blocked, error)
has_auto_expand = re.search(r"(preview_ready|blocked|error).*_expanded|_toggle", rail_js)
record("P3.6", "Auto-expands bottom rail on approval-requiring states",
       has_auto_expand is not None,
       detail="Auto-expand logic found" if has_auto_expand else "No auto-expand on approval states")

# Check for collapse/toggle functions
has_toggle = "_toggle" in rail_js and "_collapse" in rail_js
record("P3.7", "Toggle and collapse functions exist",
       has_toggle,
       detail="_toggle + _collapse found" if has_toggle else "Missing toggle/collapse functions")

p3_pass = sum(1 for r in results if r['id'].startswith('P3') and r['status'] == 'PASS')
p3_total = sum(1 for r in results if r['id'].startswith('P3'))
print(f"\nP3 complete: {p3_pass}/{p3_total} passed")

P3: YinYang Rail State Management
PASS: YinYang rail handles core FSM states (5/5) -- Found: ['idle', 'processing', 'error', 'blocked', 'done']
PASS: FSM states map to CSS classes (visual indicators) -- State-to-CSS dot mapping found
PASS: Top rail state application function exists -- _applyTopRailState found
PASS: YinYang rail listens for postMessage FSM updates -- window.message listener found
PASS: Handles yy_state message type for FSM transitions -- yy_state handler found
PASS: Auto-expands bottom rail on approval-requiring states -- Auto-expand logic found
PASS: Toggle and collapse functions exist -- _toggle + _collapse found

P3 complete: 7/7 passed


In [5]:
# P4: Tunnel page has consent flow and connection status UI
print("=" * 60)
print("P4: Tunnel Connect — Consent Flow & Status UI")
print("=" * 60)

tunnel_html = read_file(WEB / "tunnel-connect.html")

# Must have connect and disconnect buttons
has_connect_btn = 'id="connect-button"' in tunnel_html
has_disconnect_btn = 'id="disconnect-button"' in tunnel_html
record("P4.1", "Tunnel page has connect and disconnect buttons",
       has_connect_btn and has_disconnect_btn,
       detail=f"Connect={has_connect_btn}, Disconnect={has_disconnect_btn}")

# Must have status indicator (status ring or status label)
has_status = 'id="status-ring"' in tunnel_html or 'id="status-label"' in tunnel_html
record("P4.2", "Tunnel page has connection status indicator",
       has_status,
       detail="status-ring and/or status-label found" if has_status else "No status indicator")

# Must have consent/approval card explaining why tunnel is explicit
has_consent = "consent" in tunnel_html.lower() or "approval" in tunnel_html.lower()
record("P4.3", "Tunnel page has consent/approval explanation",
       has_consent,
       detail="Consent/approval language found" if has_consent else "No consent explanation")

# Fail-closed badge
has_fail_closed = "Fail-closed" in tunnel_html or "fail-closed" in tunnel_html
record("P4.4", "Tunnel page shows fail-closed default badge",
       has_fail_closed,
       detail="Fail-closed badge present" if has_fail_closed else "Missing fail-closed indicator")

# Must have endpoint display area
has_endpoint = 'id="endpoint-value"' in tunnel_html
record("P4.5", "Tunnel page has endpoint URL display area",
       has_endpoint,
       detail="endpoint-value element found" if has_endpoint else "No endpoint display")

# Should have step-by-step flow (Step 1, 2, 3)
steps = re.findall(r"Step \d", tunnel_html)
record("P4.6", "Tunnel page has multi-step consent flow",
       len(steps) >= 3,
       detail=f"Found {len(steps)} steps" if steps else "No step-by-step flow")

# Machine dashboard exists
machine_html = read_file(WEB / "machine-dashboard.html")
record("P4.7", "machine-dashboard.html exists and is non-empty",
       len(machine_html) > 100,
       detail=f"{len(machine_html)} bytes" if machine_html else "FILE MISSING")

p4_pass = sum(1 for r in results if r['id'].startswith('P4') and r['status'] == 'PASS')
p4_total = sum(1 for r in results if r['id'].startswith('P4'))
print(f"\nP4 complete: {p4_pass}/{p4_total} passed")

P4: Tunnel Connect — Consent Flow & Status UI
PASS: Tunnel page has connect and disconnect buttons -- Connect=True, Disconnect=True
PASS: Tunnel page has connection status indicator -- status-ring and/or status-label found
PASS: Tunnel page has consent/approval explanation -- Consent/approval language found
PASS: Tunnel page shows fail-closed default badge -- Fail-closed badge present
PASS: Tunnel page has endpoint URL display area -- endpoint-value element found
PASS: Tunnel page has multi-step consent flow -- Found 3 steps
PASS: machine-dashboard.html exists and is non-empty -- 5988 bytes

P4 complete: 7/7 passed


In [6]:
# P5: No hardcoded credentials or API keys in any JS file
print("=" * 60)
print("P5: No Hardcoded Credentials in JS Files")
print("=" * 60)

js_dir = JS
js_files = sorted(js_dir.glob("*.js")) if js_dir.exists() else []
record("P5.1", f"JS directory exists with files ({len(js_files)} found)",
       len(js_files) > 0,
       detail=f"{len(js_files)} JS files in web/js/")

# Patterns that indicate hardcoded secrets
secret_patterns = [
    (r'(?:api[_-]?key|apikey|secret|password|token)\s*[:=]\s*["\'][a-zA-Z0-9_\-]{20,}["\']', "Hardcoded API key/secret/token"),
    (r'sk-[a-zA-Z0-9]{20,}', "OpenAI-style API key (sk-...)"),
    (r'sk_live_[a-zA-Z0-9]+', "Stripe-style live key"),
    (r'AIza[a-zA-Z0-9_\-]{30,}', "Google API key pattern"),
    (r'ghp_[a-zA-Z0-9]{30,}', "GitHub personal access token"),
    (r'Bearer\s+[a-zA-Z0-9_\-\.]{30,}', "Hardcoded Bearer token"),
]

all_findings = []
for js_file in js_files:
    content = read_file(js_file)
    for pattern, desc in secret_patterns:
        matches = re.findall(pattern, content)
        if matches:
            all_findings.append(f"{js_file.name}: {desc} ({len(matches)} match(es))")

record("P5.2", "No hardcoded API keys or secrets in JS files",
       len(all_findings) == 0,
       detail="; ".join(all_findings) if all_findings else f"Scanned {len(js_files)} files, 0 secrets found")

# Check that auth tokens are read from localStorage, not hardcoded
cloud_js = read_file(JS / "schedule-cloud.js")
uses_localstorage = "localStorage.getItem" in cloud_js
record("P5.3", "Auth tokens read from localStorage (not hardcoded)",
       uses_localstorage,
       detail="localStorage.getItem used for token retrieval" if uses_localstorage else "No localStorage usage — tokens may be hardcoded")

# Check Authorization header uses dynamic token
has_bearer_dynamic = re.search(r"Authorization.*Bearer.*\+.*token|Authorization.*Bearer.*\$\{", cloud_js)
record("P5.4", "Authorization header uses dynamic token (not static string)",
       has_bearer_dynamic is not None,
       detail="Bearer + token concatenation found" if has_bearer_dynamic else "Check Authorization header construction")

# No console.log of tokens or secrets
token_logs = []
for js_file in js_files:
    content = read_file(js_file)
    log_matches = re.findall(r'console\.(log|info|warn)\(.*(?:token|key|secret|password).*\)', content, re.IGNORECASE)
    if log_matches:
        token_logs.append(f"{js_file.name}: {len(log_matches)} sensitive log(s)")

record("P5.5", "No console.log of tokens/keys/secrets",
       len(token_logs) == 0,
       detail="; ".join(token_logs) if token_logs else f"No sensitive logging in {len(js_files)} files")

p5_pass = sum(1 for r in results if r['id'].startswith('P5') and r['status'] == 'PASS')
p5_total = sum(1 for r in results if r['id'].startswith('P5'))
print(f"\nP5 complete: {p5_pass}/{p5_total} passed")

P5: No Hardcoded Credentials in JS Files
PASS: JS directory exists with files (16 found) -- 16 JS files in web/js/
PASS: No hardcoded API keys or secrets in JS files -- Scanned 16 files, 0 secrets found
PASS: Auth tokens read from localStorage (not hardcoded) -- localStorage.getItem used for token retrieval
PASS: Authorization header uses dynamic token (not static string) -- Bearer + token concatenation found
PASS: No console.log of tokens/keys/secrets -- No sensitive logging in 16 files

P5 complete: 5/5 passed


In [7]:
# P6: Evidence Summary — Cloud Sync QA
print("=" * 60)
print("P6: Evidence Summary — NB32 Cloud Sync QA")
print("=" * 60)

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = (passed / total * 100) if total > 0 else 0

print(f"\nTotal probes: {total}")
print(f"PASS: {passed}")
print(f"FAIL: {failed}")
print(f"Score: {score:.1f}%")
print(f"Meets minimum ({MIN_SCORE}%): {'YES' if score >= MIN_SCORE else 'NO'}")

# Per-probe-group breakdown
probe_groups = {}
for r in results:
    group = r["id"].split(".")[0]
    if group not in probe_groups:
        probe_groups[group] = {"pass": 0, "fail": 0}
    if r["status"] == "PASS":
        probe_groups[group]["pass"] += 1
    else:
        probe_groups[group]["fail"] += 1

print("\n--- Per-Group Breakdown ---")
for group, counts in sorted(probe_groups.items()):
    g_total = counts["pass"] + counts["fail"]
    g_pct = counts["pass"] / g_total * 100 if g_total else 0
    status = "PASS" if counts["fail"] == 0 else "FAIL"
    print(f"  {group}: {counts['pass']}/{g_total} ({g_pct:.0f}%) [{status}]")

if failed > 0:
    print("\n--- Failures ---")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  {r['id']}: {r['name']} -- {r['detail']}")

# Evidence hash
evidence_blob = json.dumps(results, sort_keys=True)
evidence_hash = hashlib.sha256(evidence_blob.encode()).hexdigest()[:16]
print(f"\nEvidence hash: {evidence_hash}")
print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Notebook: {NOTEBOOK_PATH}")
print(f"NORTHSTAR: {NORTHSTAR}")

assert score >= MIN_SCORE, f"Score {score:.1f}% below minimum {MIN_SCORE}%"
print(f"\n{'=' * 60}")
print(f"VERDICT: PASS ({score:.1f}% >= {MIN_SCORE}%)")
print(f"{'=' * 60}")

P6: Evidence Summary — NB32 Cloud Sync QA

Total probes: 33
PASS: 33
FAIL: 0
Score: 100.0%
Meets minimum (70%): YES

--- Per-Group Breakdown ---
  P1: 7/7 (100%) [PASS]
  P2: 7/7 (100%) [PASS]
  P3: 7/7 (100%) [PASS]
  P4: 7/7 (100%) [PASS]
  P5: 5/5 (100%) [PASS]

Evidence hash: 04e61544f2d08017
Timestamp: 2026-03-06T10:27:20.617599
Notebook: notebooks/qa/32-cloud-sync-qa.ipynb
NORTHSTAR: cloud-sync-qa

VERDICT: PASS (100.0% >= 70%)
